In [39]:
import pandas as pd
import numpy as np

In [40]:
# read the tab delimited txt file
df = pd.read_csv('305661.TXT', sep = '\t', dtype=str)

# Drop the last summary row
df = df[:-1]

print(f"Loaded {len(df)} rows")

Loaded 41 rows


In [41]:
# Clean the total (AUD) column - negatives in txt use a trailing minus
def clean_number(val):
    if pd.isna(val):
        return 0.0
    v = str(val).replace(',', '').strip()
    if v.endswith('-'):
        v = '-' + v[:-1]
    try:
        return float(v)
    except:
        return 0.0
    
df['Total (AUD)'] = df['Total (AUD)'].apply(clean_number)
df['GST (AUD)'] = df['GST (AUD)'].apply(clean_number)

In [42]:
# Add the state column
import re

def get_state(row):
    origin = str(row['Origin'])
    zone = str(row['Zone'])
    
    # Priority 1 : extract state from Origin text
    for state in ['VIC', 'NSW', 'QLD', 'SA', 'WA']:
        if state in origin:
            return state
        
    # Priority 2 : fall back to Zone code
    zone_map = {
        'YV': 'VIC',
        'YN': 'NSW',
        'YQ': 'QLD',
        'YS': 'SA',
        'YW': 'WA'
    }
    prefix = zone[:2].upper()
    return zone_map.get(prefix, '')

df['State'] = df.apply(get_state, axis = 1)

In [43]:
# Cross check
our_total = df['Total (AUD)'].sum()
our_gst = df['GST (AUD)'].sum()
our_inc_gst = our_total + our_gst

client_total = 12604.21

print(f"Our Total (ex GST):  ${our_total:,.2f}")
print(f"Our GST:             ${our_gst:,.2f}")
print(f"Our Total (inc GST): ${our_inc_gst:,.2f}")
print(f"Client Total:        ${client_total:,.2f}")
print()
if round(our_inc_gst, 2) == client_total:
    print("✓ MATCH")
else:
    print(f"⚠ DISCREPANCY of ${abs(our_inc_gst - client_total):,.2f}")

Our Total (ex GST):  $11,458.37
Our GST:             $1,145.84
Our Total (inc GST): $12,604.21
Client Total:        $12,604.21

✓ MATCH


In [44]:
# Split by Truck, then State, and write to Excel
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

wb = Workbook()
wb.remove(wb.active)  # remove default empty sheet

# Loop through each truck
for truck in df['Truck'].unique():
    truck_df = df[df['Truck'] == truck]
    
    # Loop through each state within that truck
    for state in sorted(truck_df['State'].unique()):
        state_df = truck_df[truck_df['State'] == state].copy()
        
        sheet_name = f"{truck} - {state}"
        ws = wb.create_sheet(title=sheet_name)
        
        # Write header row
        headers = list(df.columns)
        for col, header in enumerate(headers, 1):
            cell = ws.cell(row=1, column=col, value=header)
            cell.font = Font(bold=True, color='FFFFFF')
            cell.fill = PatternFill('solid', start_color='1F4E79')
        
        # Write data rows
        for row_idx, (_, row) in enumerate(state_df.iterrows(), 2):
            for col_idx, val in enumerate(row, 1):
                ws.cell(row=row_idx, column=col_idx, value=val)
        
        # Write TOTAL row
        total_row = len(state_df) + 2
        total_col = headers.index('Total (AUD)') + 1
        gst_col = headers.index('GST (AUD)') + 1
        
        ws.cell(row=total_row, column=1, value='TOTAL').font = Font(bold=True)
        ws.cell(row=total_row, column=total_col, 
                value=f'=SUM({get_column_letter(total_col)}2:{get_column_letter(total_col)}{total_row-1})').font = Font(bold=True)
        ws.cell(row=total_row, column=gst_col,
                value=f'=SUM({get_column_letter(gst_col)}2:{get_column_letter(gst_col)}{total_row-1})').font = Font(bold=True)
        
# Step 6 — Summary sheet
ws_sum = wb.create_sheet(title='Summary', index=0)  # index=0 puts it first

# Header
summary_headers = ['Truck', 'State', 'Rows', 'Total (AUD)', 'GST (AUD)', 'Total (inc GST)']
for col, header in enumerate(summary_headers, 1):
    cell = ws_sum.cell(row=1, column=col, value=header)
    cell.font = Font(bold=True, color='FFFFFF')
    cell.fill = PatternFill('solid', start_color='1F4E79')

# One row per Truck + State combination
summary_row = 2
for truck in df['Truck'].unique():
    for state in sorted(df[df['Truck'] == truck]['State'].unique()):
        state_df = df[(df['Truck'] == truck) & (df['State'] == state)]
        
        total = state_df['Total (AUD)'].sum()
        gst = state_df['GST (AUD)'].sum()
        
        ws_sum.cell(row=summary_row, column=1, value=truck)
        ws_sum.cell(row=summary_row, column=2, value=state)
        ws_sum.cell(row=summary_row, column=3, value=len(state_df))
        ws_sum.cell(row=summary_row, column=4, value=round(total, 2))
        ws_sum.cell(row=summary_row, column=5, value=round(gst, 2))
        ws_sum.cell(row=summary_row, column=6, value=round(total + gst, 2))
        summary_row += 1

# Grand total row
ws_sum.cell(row=summary_row, column=1, value='TOTAL').font = Font(bold=True)
for col in [3, 4, 5, 6]:
    ws_sum.cell(row=summary_row, column=col,
                value=f'=SUM({get_column_letter(col)}2:{get_column_letter(col)}{summary_row-1})'
                ).font = Font(bold=True)

# Client cross-check
ws_sum.cell(row=summary_row+2, column=1, value='Client Total (inc GST)')
ws_sum.cell(row=summary_row+2, column=2, value=12604.21)
ws_sum.cell(row=summary_row+3, column=1, value='Check')
ws_sum.cell(row=summary_row+3, column=2,
            value=f'=IF(F{summary_row}=B{summary_row+2},"✓ MATCH","⚠ DISCREPANCY")')

wb.save('Rohlig_split.xlsx')
print("Done — Rohlig_split.xlsx created")


Done — Rohlig_split.xlsx created
